In [1]:
import os
from pathlib import Path
from typing import Dict, List, Iterable, Optional, Tuple
import torch

In [2]:
import os
from pathlib import Path
from typing import Dict, List, Iterable, Optional, Tuple
import torch

# ---------- Checkpoint extraction ----------

def _extract_state_dict_from_loaded(obj, path: Path) -> Dict[str, torch.Tensor]:
    # raw state_dict
    if isinstance(obj, dict) and all(isinstance(v, torch.Tensor) for v in obj.values()):
        return obj
    # dict with common nesting
    if isinstance(obj, dict):
        if "state_dict" in obj and isinstance(obj["state_dict"], dict):
            return obj["state_dict"]
        if "model" in obj and hasattr(obj["model"], "state_dict"):
            return obj["model"].state_dict()
    # nn.Module
    if hasattr(obj, "state_dict") and callable(getattr(obj, "state_dict")):
        try:
            return obj.state_dict()
        except Exception:
            pass
    # your ImplicitRegistrator with .network
    if hasattr(obj, "network") and hasattr(obj.network, "state_dict"):
        return obj.network.state_dict()
    # optional reverse
    if hasattr(obj, "network_rev") and hasattr(obj.network_rev, "state_dict"):
        return obj.network_rev.state_dict()
    raise TypeError(f"Unknown checkpoint format at {path} (type={type(obj)})")

def load_state_dict(path: Path, model_name: str = "model_network.pth") -> Dict[str, torch.Tensor]:
    obj = torch.load(path / model_name, map_location="cpu")
    sd = _extract_state_dict_from_loaded(obj, path)
    # Normalize dtype
    sd = {k: (v.float() if isinstance(v, torch.Tensor) and v.dtype.is_floating_point else v)
          for k, v in sd.items()}
    return sd

# ---------- Averaging helpers ----------

def _check_key_compat(sd_list: List[Dict[str, torch.Tensor]]) -> List[str]:
    key_sets = [set(sd.keys()) for sd in sd_list]
    base = key_sets[0]
    for i, ks in enumerate(key_sets[1:], start=1):
        if ks != base:
            missing = base - ks
            extra = ks - base
            msg = []
            if missing:
                msg.append(f"missing in #{i}: {sorted(list(missing))[:8]}")
            if extra:
                msg.append(f"extra in #{i}: {sorted(list(extra))[:8]}")
            raise ValueError("State dict key mismatch: " + " | ".join(msg))
    return sorted(list(base))

def _winsorize_stack(tensors: List[torch.Tensor],
                     lower_pct: float,
                     upper_pct: float) -> List[torch.Tensor]:
    stacked = torch.stack(tensors, dim=0)
    lo = torch.quantile(stacked, lower_pct/100.0, dim=0)
    hi = torch.quantile(stacked, upper_pct/100.0, dim=0)
    clamped = torch.clamp(stacked, min=lo, max=hi)
    return [clamped[i] for i in range(clamped.shape[0])]

def average_state_dicts(sd_list: List[Dict[str, torch.Tensor]],
                        winsorize: Optional[Tuple[float, float]] = None
                       ) -> Dict[str, torch.Tensor]:
    keys = _check_key_compat(sd_list)
    avg: Dict[str, torch.Tensor] = {}
    for k in keys:
        vals = [sd[k] for sd in sd_list]
        if isinstance(vals[0], torch.Tensor) and vals[0].dtype.is_floating_point:
            if winsorize is not None:
                l,u = winsorize
                vals = _winsorize_stack(vals, l, u)
            stacked = torch.stack(vals, dim=0)
            avg[k] = stacked.mean(dim=0)
        else:
            if isinstance(vals[0], torch.Tensor) and vals[0].dtype in (torch.int32, torch.int64):
                stacked = torch.stack(vals, dim=0)
                avg[k] = stacked.float().mean(dim=0).round().to(vals[0].dtype)
            else:
                avg[k] = vals[0]
    return avg

In [3]:
from pathlib import Path
import torch

def debug_load_some_networks(
    root: str,
    model_name: str = "model_network.pth",
    n_samples: int = 3
):
    root_path = Path(root)

    # Get all subdirectories = PIDs
    pid_dirs = sorted([p for p in root_path.iterdir() if p.is_dir()])
    print(f"Found {len(pid_dirs)} patient folders under {root_path}")

    loaded = {}

    count = 0
    errors = 0
    for pid_dir in pid_dirs:
        model_path = pid_dir / model_name
        #print(f"Checking {model_path}...")  
        if not model_path.exists():
            # print(f"[WARN] No model for {pid_dir.name}")
            errors += 1
            continue

        try:
            ckpt = torch.load(model_path, map_location="cpu")

            # unwrap possibility
            if isinstance(ckpt, dict) and "state_dict" in ckpt:
                sd = ckpt["state_dict"]
            else:
                sd = ckpt

            print(f"[{count}] Loaded {model_path}")
            print(f"     keys: {list(sd.keys())[:5]} ... ({len(sd)} params)")
            loaded[pid_dir.name] = model_path
            count += 1

            if count >= n_samples:
                break

        except Exception as e:
            print(f"[ERROR] Failed to load {model_path}: {e}")
            errors += 1

    print(f"Finished loading. Successfully loaded {count} models with {errors} errors.")
    
    return loaded


In [4]:
from pathlib import Path
from typing import Optional, Tuple, Dict, List
import torch
import csv

def build_priors(
    root: str,
    out_dir: str,
    model_name: str = "model_network.pth",
    winsorize_pct: Optional[Tuple[float, float]] = None,
    sample_size: Optional[int] = None,   # e.g. 50, 100, 200; None => use all valid
    out_name_prefix: str = "prior_init",
    verbose: bool = False
) -> Dict[str, object]:

    def vprint(*args, **kwargs):
        if verbose:
            print(*args, **kwargs)

    root_path = Path(root)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # --- gather PID dirs with model files ---
    pid_dirs = sorted([p for p in root_path.iterdir() if p.is_dir()])
    pid_model_pairs = []

    for pid_dir in pid_dirs:
        model_path = pid_dir / model_name
        if model_path.exists():
            pid_model_pairs.append((pid_dir.name, model_path))
        else:
            vprint(f"[WARN] No model '{model_name}' for PID={pid_dir.name}")

    if not pid_model_pairs:
        vprint("[ERROR] No model paths found. Aborting.")
        return {}

    total_candidates = len(pid_model_pairs)
    vprint(f"Found {total_candidates} candidate model files named '{model_name}'")

    # --- attempt loading ALL models first ---
    valid_models: List[Tuple[str, dict]] = []
    failed: List[Tuple[str, str]] = []

    for pid, model_path in pid_model_pairs:
        try:
            ckpt = torch.load(model_path, map_location="cpu")
            if isinstance(ckpt, dict) and "state_dict" in ckpt:
                sd = ckpt["state_dict"]
            else:
                sd = ckpt

            valid_models.append((pid, sd))
            vprint(f"[OK] PID={pid} -> loaded (|params|={len(sd)})")

        except Exception as e:
            msg = str(e)
            failed.append((pid, msg))
            vprint(f"[WARN] Skipping PID={pid}: {msg}")

    if not valid_models:
        vprint("[ERROR] No models could be loaded successfully. Aborting.")
        return {}

    total_valid = len(valid_models)
    total_failed = len(failed)
    print(f"[INFO] Valid models = {total_valid}, Failed = {total_failed}")

    # --- apply sample size only to valid models ---
    if sample_size is None or sample_size >= total_valid:
        n_use = total_valid
    else:
        n_use = sample_size

    if sample_size is not None and sample_size > total_valid:
        print(f"[WARN] Requested sample_size={sample_size}, "
               f"but only {total_valid} valid models available. Using all valid.")

    # deterministic slice
    valid_models = valid_models[:n_use]
    used_pids = [pid for pid, _ in valid_models]
    sd_list = [sd for _, sd in valid_models]

    print(f"[INFO] Using {len(sd_list)} valid models to build the prior.")

    # --- average ---
    try:
        avg_sd = average_state_dicts(sd_list, winsorize=winsorize_pct)
    except ValueError as e:
        vprint(f"[ERROR] Averaging failed: {e}")
        return {}

    actual_n_used = len(sd_list)
    out_name = f"{out_name_prefix}_N{actual_n_used}.pth"
    out_path = out_dir / out_name

    torch.save(avg_sd, out_path)
    print(f"[OK] saved prior -> {out_path}")

    # --- Save CSV of PIDs used ---
    csv_name = f"{out_name_prefix}_N{actual_n_used}_pids.csv"
    csv_path = out_dir / csv_name

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["pid"])
        for pid in used_pids:
            writer.writerow([pid])

    print(f"[OK] saved PID list -> {csv_path}")

    return {
        "prior_path": out_path,
        "pid_csv": csv_path,
        "n_used": actual_n_used,
        "used_pids": used_pids,
        "n_valid": total_valid,
        "n_failed": total_failed,
    }


In [ ]:
path_to_experiments = ""
selected_experiment = ""  
path_to_experiment = os.path.join(path_to_experiments, selected_experiment)
out_prior_dir = os.path.join(path_to_experiment, "priors")
out_prior_dir_canonical = os.path.join(out_prior_dir, "priors_canonical_winsor")

# loaded = debug_load_some_networks(
#     root=path_to_experiment,
#     model_name="final_model.pth",
#     n_samples=3,
# )
# print("Loaded sample:", loaded)

In [ ]:
out = build_priors(
    root=path_to_experiment,
    out_dir=out_prior_dir_canonical,
    model_name="final_model.pth",
    winsorize_pct=(1.0, 99.0),
    sample_size=750,
)